# IIASA API Endpoint Data Format Exploration

This notebook systematically queries all major endpoints of the IIASA Scenario Compass API to understand and document the data format and structure of each endpoint's output.

In [2]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [3]:
import pandas as pd  # noqa: F401
import numpy as np
import json
from pprint import pprint
import sys
import os
from pyam import IamDataFrame, Statistics, filter_by_meta  # noqa: F401

sys.path.append('..')
from src.iasa.iiasa_client import IIASABaseAPIClient
from src.iasa.utils import create_iamdf, create_df_from_api_response

[WARNING] 20:19:58 - pint.util: Redefining 'kt' (<class 'pint.delegates.txt_defparser.plain.UnitDefinition'>)
[WARNING] 20:19:58 - pint.util: Redefining 'EUR_2005' (<class 'pint.delegates.txt_defparser.plain.UnitDefinition'>)
[WARNING] 20:19:58 - pint.util: Redefining 'EUR' (<class 'pint.delegates.txt_defparser.plain.UnitDefinition'>)


In [4]:
client = IIASABaseAPIClient()

## Helper Functions for Analysis

## Major Endpoints Analysis

Based on the documentation, the major endpoints are:
- runs/
- regions/
- variables/
- meta/
- models/
- scenarios/
- iamc/datapoints/

### 1. Runs Endpoint

In [5]:
runs_df = create_df_from_api_response(client.get_data(
    "runs/",
    params={"table": "true"}
))
runs_df

,model__id,scenario__id,version,is_default,id,updated_at,updated_by,created_at,created_by
0,1,1,1,True,1,2025-03-28 12:04:39.091252,@unknown,2025-03-28 12:04:24.885658,@unknown
1,1,2,1,True,2,2025-03-28 12:04:45.359392,@unknown,2025-03-28 12:04:39.918461,@unknown
2,1,3,1,True,3,2025-03-28 12:04:52.187804,@unknown,2025-03-28 12:04:46.039148,@unknown
3,1,4,1,True,4,2025-03-28 12:04:58.703293,@unknown,2025-03-28 12:04:52.911709,@unknown
4,1,5,1,True,5,2025-03-28 12:05:05.292936,@unknown,2025-03-28 12:04:59.544864,@unknown
...,...,...,...,...,...,...,...,...,...
971,30,304,1,True,972,2025-07-28 15:57:57.529314,@unknown,2025-07-28 15:57:46.241788,@unknown
972,30,305,1,True,973,2025-07-28 15:58:08.842858,@unknown,2025-07-28 15:57:57.793507,@unknown
973,30,306,1,True,974,2025-07-28 15:58:20.881684,@unknown,2025-07-28 15:58:09.036415,@unknown
974,30,307,1,True,975,2025-07-28 15:58:36.182289,@unknown,2025-07-28 15:58:21.134060,@unknown


### 2. Regions Endpoint

In [6]:
runs_df = create_df_from_api_response(client.get_data(
    "regions/",
    params={"table": "true"}
))
runs_df

,name,hierarchy,id,created_at,created_by
0,World,common,318,2025-03-20 15:11:06.387377,@unknown
1,International,common,320,2025-03-28 11:50:38.561727,@unknown
2,OECD & EU (R5),R5,321,2025-03-28 11:50:38.810880,@unknown
3,Reforming Economies (R5),R5,322,2025-03-28 11:50:38.858697,@unknown
4,Asia (R5),R5,323,2025-03-28 11:50:38.909370,@unknown
...,...,...,...,...,...
597,REMIND-MAgPIE 3.5-4.11|Russia and Reforming Ec...,REMIND-MAgPIE 3.5-4.11,916,2025-07-14 16:40:33.673106,@unknown
598,REMIND-MAgPIE 3.5-4.11|Sub-Saharan Africa,REMIND-MAgPIE 3.5-4.11,917,2025-07-14 16:40:33.703406,@unknown
599,REMIND-MAgPIE 3.5-4.11|United States of America,REMIND-MAgPIE 3.5-4.11,918,2025-07-14 16:40:33.731659,@unknown
600,REMIND-MAgPIE 3.5-4.11|EU 28,REMIND-MAgPIE 3.5-4.11,919,2025-07-14 16:40:33.771070,@unknown


### 3. Variables Endpoint

In [7]:
variables = create_df_from_api_response(client.get_data(
    "iamc/variables/",
    params={"table": "true"}
))
variables

,name,id,created_at,created_by
0,Agricultural Demand,1,2025-03-28T12:04:25.352595,@unknown
1,Agricultural Production|Crops|Energy Crops,2,2025-03-28T12:04:25.396617,@unknown
2,Capacity Additions|Electricity|Biomass,3,2025-03-28T12:04:25.436362,@unknown
3,Capacity Additions|Electricity|Coal,4,2025-03-28T12:04:25.475249,@unknown
4,Capacity Additions|Electricity|Gas,5,2025-03-28T12:04:25.513302,@unknown
...,...,...,...,...
1845,Price|Carbon [weighted by Final Energy],1886,2025-07-11T12:10:30.733336,@unknown
1846,Capacity|Electricity|Fossil,1887,2025-07-25T20:28:26.472890,@unknown
1847,Capacity|Electricity|Non-Biomass Renewables,1888,2025-07-25T20:28:26.535393,@unknown
1848,Primary Energy|Biomass|Modern|w/ CCS,1889,2025-07-25T20:28:26.720469,@unknown


### 4. Models Endpoint

In [17]:
models = create_df_from_api_response(
    client.get_data(
        "models/",
        params={
            "join_parameters": "true",
            "join_runs": "true",
            "table": "true",
        },
    )
)

models

,name,id,created_at,created_by
0,AIM/CGE V2.2,1,2025-03-28 12:04:24.863850,@unknown
1,MESSAGEix-GLOBIOM 1.1,2,2025-03-28 12:40:19.727393,@unknown
2,IMAGE 3.0,3,2025-03-28 12:59:05.024707,@unknown
3,WITCH 5.0,4,2025-03-28 13:22:19.746918,@unknown
4,COFFEE 1.1,5,2025-03-28 13:34:43.084588,@unknown
5,GEM-E3 V2021,6,2025-03-28 14:11:53.838058,@unknown
6,POLES-JRC ENGAGE,7,2025-03-28 16:44:53.587610,@unknown
7,REMIND-MAgPIE 2.1-4.2,8,2025-04-02 08:18:31.620923,@unknown
8,TIAM-ECN 1.1,9,2025-04-02 10:21:00.357146,@unknown
9,REMIND 3.0,10,2025-04-07 07:41:37.045853,@unknown


### 5. Scenarios Endpoint

In [13]:
scenarios = create_df_from_api_response(client.get_data(
    "scenarios/",
    params={"table": "true"}
))
scenarios

,name,id,created_at,created_by
0,ENGAGE-INDCi2030-1000f,1,2025-03-28 12:04:24.883425,@unknown
1,ENGAGE-INDCi2030-1200,2,2025-03-28 12:04:39.918461,@unknown
2,ENGAGE-INDCi2030-1200f,3,2025-03-28 12:04:46.039148,@unknown
3,ENGAGE-INDCi2030-1400,4,2025-03-28 12:04:52.911709,@unknown
4,ENGAGE-INDCi2030-1400f,5,2025-03-28 12:04:59.544864,@unknown
...,...,...,...,...
303,Deep-Mitigation-MAC_99_n8,304,2025-07-28 15:57:46.241788,@unknown
304,Deep-Mitigation-SSP2,305,2025-07-28 15:57:57.793507,@unknown
305,Deep-Mitigation-SSP3,306,2025-07-28 15:58:09.036415,@unknown
306,Deep-Mitigation-SSP4,307,2025-07-28 15:58:21.123980,@unknown


### 6. Meta Endpoint

In [12]:
meta = create_df_from_api_response(client.get_data(
    "meta/",
    params={"table": "true"}
))
meta

,run__id,key,id,value,type
0,1,Project,1,H2020 ENGAGE,STR
1,1,Scientific Manuscript (Citation),2,Riahi et al. (2021),STR
2,1,Scientific Manuscript (DOI),3,10.1038/s41558-021-01215-2,STR
3,1,Data Source (DOI),4,10.5281/zenodo.5553976,STR
4,2,Project,5,H2020 ENGAGE,STR
...,...,...,...,...,...
49397,912,Climate Assessment|Category [Name],81032,C1b: Below 1.5°C with low OS,STR
49398,912,Climate Assessment|Processing|Harmonization,81033,aneris (version: 0.3.1),STR
49399,912,Climate Assessment|Processing|Infilling,81034,silicone (version: 1.3.0),STR
49400,912,Climate Assessment|Processing|Runner,81035,openscm_runner (version: 0.12.1),STR
